# WAAM Test 데이터 노이즈 증강 — Split(MOL+INC) 구조 기반

## 대상 데이터
```
D:\code\Artificial intelligence\Split\MOL\split\split\test\0, \1
D:\code\Artificial intelligence\Split\INC\split\split\test\0, \1
```
위 4개 폴더(MOL/test/0,1 + INC/test/0,1)의 **test 데이터에만** 노이즈 증강을 적용한다.

## 노이즈 4종 (공정 중 발생하는 실제 노이즈)

| 노이즈 | 설명 |
|--------|------|
| **Arc Glare (빛번짐)** | 아크 발생 시 카메라 포화(saturation), 이미지 흰색 뭉개짐 |
| **Spatter (스패터)** | 금속 용융물 튀김 → 이미지에 밝은 점/잡음 |
| **Fume/Smoke (연기)** | 용접 흄이 시야를 가림 |
| **Vibration (흔들림)** | 로봇 암 진동 → 블러, 위치 오차 |

## 증강 방법 6가지

| # | 방법 | 구성 |
|---|------|------|
| 1 | Arc Glare | 빛번짐 단독 |
| 2 | Spatter | 스패터 단독 |
| 3 | Fume/Smoke | 연기 단독 |
| 4 | Vibration | 흔들림 단독 |
| 5 | Random-2 Mixed | 4종 중 2개를 랜덤으로 뽑아 조합 |
| 6 | Full Mixed | 4종 전체 조합 |

> ✅ RGB(3채널) 기준으로 처리, Grayscale 미사용

## 1. 라이브러리 임포트

In [ ]:
import cv2
import numpy as np
from PIL import Image
import os
import random
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

print('라이브러리 로드 완료')

## 2. 경로 및 설정값 (Split 구조 — MOL + INC, Test만 대상)

In [ ]:
# ============================================================
# ✅ Split 데이터 경로 설정 (Test 데이터만 증강 대상)
# ============================================================
MOL_SPLIT_DIR = r'D:\code\Artificial intelligence\Split\MOL\split\split'   # 몰리브덴
INC_SPLIT_DIR = r'D:\code\Artificial intelligence\Split\INC\split\split'   # 인코넬

SOURCE_DIRS = {
    'MOL': MOL_SPLIT_DIR,
    'INC': INC_SPLIT_DIR,
}

TARGET_SPLIT  = 'test'        # 증강 대상 split (test 고정)
LABEL_FOLDERS = ['0', '1']    # 0=정상, 1=불량 — 둘 다 증강 대상

# 증강 결과 저장 위치: 원본 Split 구조와 분리된 별도 출력 폴더
AUG_OUTPUT_DIR = r'D:\code\Artificial intelligence\Split\test_augmented'

IMG_EXTENSIONS = ('.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.tif')
# ============================================================

os.makedirs(AUG_OUTPUT_DIR, exist_ok=True)

# ── 경로 존재 여부 점검 ──
print('경로 점검 (Test 데이터만 대상):')
for src_name, src_dir in SOURCE_DIRS.items():
    for label in LABEL_FOLDERS:
        folder = os.path.join(src_dir, TARGET_SPLIT, label)
        exists = os.path.isdir(folder)
        n_files = len(os.listdir(folder)) if exists else 0
        status = '✅' if exists else '❌ 없음'
        print(f'  [{src_name}] {TARGET_SPLIT}/{label:1s} → {status}  ({n_files}장)  {folder}')

print(f'\n증강 결과 저장 경로: {AUG_OUTPUT_DIR}')

## 3. 노이즈 함수 정의 (4종)

### 3-1. Arc Glare (빛번짐)
아크 방전 시 카메라 포화(saturation) → 이미지 중심부가 흰색으로 뭉개지는 현상

In [ ]:
def add_arc_glare(img_rgb, n_glare=(1, 3), radius_ratio=(0.05, 0.2), intensity=(0.6, 1.0)):
    """
    Arc Glare (빛번짐)
    - 아크 위치(이미지 하단 중심)에 강한 흰색 원형 광원 추가
    - 중심 → 외곽으로 가우시안 페이드아웃
    - RGB 3채널 모두 동일하게 적용 (흰색 빛)
    """
    result = img_rgb.astype(np.float32).copy()
    h, w   = img_rgb.shape[:2]
    n      = random.randint(*n_glare)

    for _ in range(n):
        cx     = random.randint(int(w * 0.3), int(w * 0.7))
        cy     = random.randint(int(h * 0.5), int(h * 0.9))
        radius = random.uniform(*radius_ratio) * min(h, w)
        alpha  = random.uniform(*intensity)

        y_grid, x_grid = np.ogrid[:h, :w]
        dist  = np.sqrt((x_grid - cx)**2 + (y_grid - cy)**2).astype(np.float32)

        sigma = radius / 2.5
        glare = np.exp(-dist**2 / (2 * sigma**2)) * alpha * 255
        glare = glare[:, :, np.newaxis]

        result = np.clip(result + glare, 0, 255)

    return result.astype(np.uint8)

### 3-2. Spatter (스패터)
금속 용융물 튀김 → 이미지에 밝은 점/잡음이 불규칙하게 분포

In [ ]:
def add_spatter(img_rgb, n_range=(10, 60), size_range=(1, 5), brightness=(200, 255)):
    """
    Spatter (스패터)
    - 랜덤 위치에 작은 밝은 원형 점 다수 생성
    - 용융물이 튀는 방향 특성상 하단 절반에 집중
    - 일부는 꼬리(trail) 형태로 연장
    """
    result = img_rgb.copy()
    h, w   = img_rgb.shape[:2]
    n      = random.randint(*n_range)

    for _ in range(n):
        cx   = random.randint(0, w - 1)
        cy   = random.randint(h // 3, h - 1)
        r    = random.randint(*size_range)
        br   = random.randint(*brightness)
        color = (br, int(br * random.uniform(0.8, 1.0)), int(br * random.uniform(0.5, 0.8)))
        cv2.circle(result, (cx, cy), r, color, -1)

        if random.random() < 0.5:
            trail_len = random.randint(3, 15)
            angle     = random.uniform(0, 2 * np.pi)
            ex = int(cx + trail_len * np.cos(angle))
            ey = int(cy + trail_len * np.sin(angle))
            ex = np.clip(ex, 0, w - 1)
            ey = np.clip(ey, 0, h - 1)
            cv2.line(result, (cx, cy), (ex, ey), color, 1)

    return result

### 3-3. Fume/Smoke (연기)
용접 흄이 시야를 가림 → 반투명 연기층 (멀티스케일 프랙탈 노이즈 기반)

In [ ]:
def add_fume_smoke(img_rgb, intensity_range=(0.05, 0.2), upper_bias=True):
    """
    Fume/Smoke (연기)
    - 멀티스케일 프랙탈 노이즈로 자연스러운 연기 텍스처 생성
    - upper_bias=True: 연기는 위로 올라가므로 상단에 더 강하게 적용
    - RGB 3채널에 동일한 연기 레이어 적용 (회백색 연기)
    """
    h, w      = img_rgb.shape[:2]
    intensity = random.uniform(*intensity_range)

    smoke = np.zeros((h, w), dtype=np.float32)
    for scale in [0.02, 0.05, 0.1, 0.2]:
        sh = max(1, int(h * scale))
        sw = max(1, int(w * scale))
        layer = np.random.rand(sh, sw).astype(np.float32)
        layer = cv2.resize(layer, (w, h), interpolation=cv2.INTER_LINEAR)
        smoke += layer

    smoke = (smoke - smoke.min()) / (smoke.max() - smoke.min() + 1e-8)

    if upper_bias and random.random() < 0.6:
        mask = np.ones((h, w), dtype=np.float32)
        mask[:h//2] *= random.uniform(1.1, 1.5)
        smoke = np.clip(smoke * mask, 0, 1)

    smoke_layer = (smoke * 255 * intensity).astype(np.float32)
    smoke_rgb   = np.stack([smoke_layer] * 3, axis=-1)
    result      = np.clip(img_rgb.astype(np.float32) + smoke_rgb, 0, 255)

    return result.astype(np.uint8)

### 3-4. Vibration (흔들림)
로봇 암 진동 → 모션 블러 + 랜덤 픽셀 오프셋(위치 오차)

In [ ]:
def add_vibration(img_rgb, blur_ksize_range=(3, 9), shift_range=(-8, 8)):
    """
    Vibration (흔들림)
    - 방향성 모션 블러: 수평/수직/대각선 방향 랜덤 선택
    - 랜덤 픽셀 오프셋(affine shift): 로봇 진동으로 인한 위치 오차
    - RGB 3채널 동일하게 적용
    """
    h, w   = img_rgb.shape[:2]
    result = img_rgb.copy()

    ksize    = random.choice([k for k in range(blur_ksize_range[0], blur_ksize_range[1]+1, 2)])
    kernel   = np.zeros((ksize, ksize), dtype=np.float32)
    blur_dir = random.choice(['horizontal', 'vertical', 'diagonal'])

    if blur_dir == 'horizontal':
        kernel[ksize//2, :] = 1.0 / ksize
    elif blur_dir == 'vertical':
        kernel[:, ksize//2] = 1.0 / ksize
    else:
        np.fill_diagonal(kernel, 1.0 / ksize)

    result = cv2.filter2D(result, -1, kernel)

    dx = random.randint(*shift_range)
    dy = random.randint(*shift_range)
    M  = np.float32([[1, 0, dx], [0, 1, dy]])
    result = cv2.warpAffine(result, M, (w, h),
                            borderMode=cv2.BORDER_REFLECT)

    return result

## 4. 증강 방법 6가지 정의
단독 4종 + 랜덤 2개 조합 + 전체 조합

In [ ]:
# ============================================================
# ✅ 노이즈 함수 매핑 (4종)
# ============================================================
NOISE_FUNCTIONS = {
    'arc_glare' : add_arc_glare,
    'spatter'   : add_spatter,
    'fume_smoke': add_fume_smoke,
    'vibration' : add_vibration,
}
NOISE_NAMES = list(NOISE_FUNCTIONS.keys())  # ['arc_glare', 'spatter', 'fume_smoke', 'vibration']


def apply_noise_chain(img_rgb, noise_keys):
    """주어진 노이즈 함수 리스트를 순서대로 적용"""
    result = img_rgb.copy()
    for key in noise_keys:
        result = NOISE_FUNCTIONS[key](result)
    return result


def augment_method_1_arc_glare(img):
    return apply_noise_chain(img, ['arc_glare']), 'arc_glare'

def augment_method_2_spatter(img):
    return apply_noise_chain(img, ['spatter']), 'spatter'

def augment_method_3_fume_smoke(img):
    return apply_noise_chain(img, ['fume_smoke']), 'fume_smoke'

def augment_method_4_vibration(img):
    return apply_noise_chain(img, ['vibration']), 'vibration'

def augment_method_5_random_two(img):
    """4종 중 2개를 랜덤으로 뽑아 조합"""
    picked = random.sample(NOISE_NAMES, 2)
    result = apply_noise_chain(img, picked)
    tag = 'random2_' + '_'.join(picked)
    return result, tag

def augment_method_6_full_mixed(img):
    """4종 전체 조합"""
    return apply_noise_chain(img, NOISE_NAMES), 'full_mixed'


AUGMENT_METHODS = {
    1: augment_method_1_arc_glare,
    2: augment_method_2_spatter,
    3: augment_method_3_fume_smoke,
    4: augment_method_4_vibration,
    5: augment_method_5_random_two,
    6: augment_method_6_full_mixed,
}

METHOD_LABELS = {
    1: 'Arc Glare (빛번짐)',
    2: 'Spatter (스패터)',
    3: 'Fume/Smoke (연기)',
    4: 'Vibration (흔들림)',
    5: 'Random-2 Mixed (2개 랜덤 조합)',
    6: 'Full Mixed (전체 조합)',
}

print('증강 방법 6가지:')
for k, v in METHOD_LABELS.items():
    print(f'  {k}. {v}')

## 5. Test 데이터 전체 증강 실행 (MOL + INC, 라벨 0+1, 6가지 방법 각각 적용)

In [ ]:
AUG_CLEAN_BEFORE_RERUN = True   # 재실행 시 기존 증강 결과 삭제 후 새로 생성 (중복 방지)


def run_test_augmentation(source_dirs, target_split, label_folders,
                          augment_methods, method_labels, output_dir,
                          clean_before_rerun=True):
    """
    MOL/INC 의 test/{0,1} 데이터 전체에 대해
    6가지 증강 방법을 각각 적용하여 별도 출력 폴더에 저장.

    출력 구조:
      output_dir/
        {method_num}_{method_name}/
          {source}_{label}/
            원본파일명_aug.png
    """
    summary = {}

    for method_num, method_fn in augment_methods.items():
        method_tag = method_fn.__name__.replace('augment_method_', '')
        method_dir = os.path.join(output_dir, f'{method_num}_{method_tag}')

        if clean_before_rerun and os.path.isdir(method_dir):
            import shutil
            shutil.rmtree(method_dir)
        os.makedirs(method_dir, exist_ok=True)

        total_saved = 0
        for src_name, src_dir in source_dirs.items():
            for label in label_folders:
                in_folder = os.path.join(src_dir, target_split, label)
                if not os.path.isdir(in_folder):
                    continue

                out_folder = os.path.join(method_dir, f'{src_name}_{label}')
                os.makedirs(out_folder, exist_ok=True)

                img_files = [f for f in os.listdir(in_folder) if f.lower().endswith(IMG_EXTENSIONS)]

                for fname in tqdm(img_files, desc=f'[{method_num}.{method_tag}] {src_name}/{label}', leave=False):
                    img_path = os.path.join(in_folder, fname)
                    img_rgb  = np.array(Image.open(img_path).convert('RGB'))

                    aug_img, applied_tag = method_fn(img_rgb)

                    stem      = os.path.splitext(fname)[0]
                    save_name = f'{stem}_{applied_tag}.png'
                    Image.fromarray(aug_img).save(os.path.join(out_folder, save_name))
                    total_saved += 1

        summary[f'{method_num}_{method_tag}'] = total_saved
        print(f'[{method_num}] {method_labels[method_num]:35s} → {total_saved}장 저장 ({method_dir})')

    return summary


from tqdm import tqdm

aug_summary = run_test_augmentation(
    SOURCE_DIRS, TARGET_SPLIT, LABEL_FOLDERS,
    AUGMENT_METHODS, METHOD_LABELS, AUG_OUTPUT_DIR,
    clean_before_rerun=AUG_CLEAN_BEFORE_RERUN
)

print(f'\n✅ 전체 증강 완료! 총 {sum(aug_summary.values())}장 생성됨')
print(f'저장 위치: {AUG_OUTPUT_DIR}')

## 6. 증강 결과 미리보기 — 6가지 방법 비교

In [ ]:
def preview_six_methods(source_dirs, target_split, n_samples=3,
                        save_path='test_augmentation_preview.png'):
    """
    원본 1장에 대해 6가지 증강 방법을 모두 적용한 결과를 나란히 비교.
    """
    # 샘플 이미지 수집 (MOL/INC, 라벨 0/1 섞어서)
    candidates = []
    for src_name, src_dir in source_dirs.items():
        for label in LABEL_FOLDERS:
            folder = os.path.join(src_dir, target_split, label)
            if not os.path.isdir(folder):
                continue
            files = [f for f in os.listdir(folder) if f.lower().endswith(IMG_EXTENSIONS)]
            candidates.extend([(folder, f) for f in files])

    samples = random.sample(candidates, min(n_samples, len(candidates)))

    n_cols = 7  # 원본 + 6가지 방법
    fig, axes = plt.subplots(n_samples, n_cols,
                             figsize=(n_cols * 3.0, n_samples * 3.0))
    if n_samples == 1:
        axes = axes[np.newaxis, :]

    fig.patch.set_facecolor('#1a1a2e')

    col_titles = ['원본'] + [METHOD_LABELS[i] for i in range(1, 7)]
    col_colors = ['#ffffff', '#ffcc44', '#ff8844', '#88ddaa', '#88aaff', '#cc88ff', '#ff4466']

    for j, (title, color) in enumerate(zip(col_titles, col_colors)):
        axes[0, j].set_title(title, fontsize=8, color=color, fontweight='bold', pad=6)

    for i, (folder, fname) in enumerate(samples):
        img = np.array(Image.open(os.path.join(folder, fname)).convert('RGB'))

        axes[i, 0].imshow(img, aspect='auto')
        axes[i, 0].axis('off')
        axes[i, 0].patch.set_facecolor('#1a1a2e')

        for j in range(1, 7):
            aug, _ = AUGMENT_METHODS[j](img)
            axes[i, j].imshow(aug, aspect='auto')
            axes[i, j].axis('off')
            axes[i, j].patch.set_facecolor('#1a1a2e')

        axes[i, 0].set_ylabel(f'샘플 {i+1}', fontsize=9, color='#aaaaaa', labelpad=6)
        axes[i, 0].yaxis.set_label_position('left')
        axes[i, 0].yaxis.label.set_visible(True)

    fig.suptitle('WAAM Test 데이터 노이즈 증강 6가지 비교', fontsize=14,
                 color='white', fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()
    print(f'저장 완료: {save_path}')


preview_six_methods(SOURCE_DIRS, TARGET_SPLIT, n_samples=3,
                    save_path='test_augmentation_preview.png')